# カワウソ先生 — Colab で動画を書き出すターミナルは使いません。上から順に ▶ を押すだけです。| セル | やること | かかる時間 ||---|---|---|| 1 | 下ごしらえ（1回だけ） | 2〜3分 || 2 | 書き出し | 20〜30分 || 3 | 見る・落とす | すぐ |**先に1回だけ**：左の 🔑 から `GEMINI_API_KEY` を登録して、ノートブックからのアクセスを ON にしてください。

## 1. 下ごしらえ

In [ ]:
#@title ▶ を押す（2〜3分）import os, subprocess, sysBR = 'claude/otter-video-monetization-frvk65'# 日本語フォント。style.py が探す順に合わせる（Noto Serif と M+）print('フォントを入れています...')subprocess.run('apt-get -qq install -y fonts-noto-cjk fonts-mplus',               shell=True, capture_output=True)print('ライブラリを入れています...')subprocess.run(f'{sys.executable} -m pip -q install imageio-ffmpeg pillow numpy',               shell=True)# 履歴なしで取ってくる。管理下は118MBほどif not os.path.isdir('/content/claude-mcp'):    print('リポジトリを取ってきています...')    subprocess.run(f'git clone -q --depth 1 -b {BR} '                   'https://github.com/masa186/claude-mcp.git /content/claude-mcp',                   shell=True)else:    subprocess.run(f'cd /content/claude-mcp && git fetch -q --depth 1 origin {BR} '                   f'&& git reset -q --hard origin/{BR}', shell=True)# API キーは Colab の 🔑 から。声はキャッシュにあるので、# 台本を書き換えなければ呼び出しは起きないtry:    from google.colab import userdata    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')    print('APIキー: 読めました')except Exception as e:    print('APIキーが読めません（台本を変えないなら無くても書き出せます）:', e)os.chdir('/content/claude-mcp/pipeline')import styleprint()print('フォント:', style.SERIF[0].split('/')[-1], '/', style.GOTHIC[0].split('/')[-1])print('下ごしらえ完了')

## 2. 書き出し`EP` の数字を変えれば他の回も出せます。`SHORT = True` にすると TikTok 用の短縮版になります。

In [ ]:
#@title ▶ を押す（20〜30分。タブは開いたままに）EP    = 7      #@param {type:"integer"}SHORT = False  #@param {type:"boolean"}import os, subprocess, sys, timeos.chdir('/content/claude-mcp/pipeline')env = dict(os.environ, PYTHONUNBUFFERED='1')if SHORT:    env['SHORT'] = '1't0 = time.time()p = subprocess.Popen([sys.executable, f'ep{EP:02d}.py'], env=env,                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)for line in p.stdout:    print(line.rstrip(), flush=True)p.wait()print('\n%.0f分かかりました' % ((time.time()-t0)/60))

## 3. 見る・落とす

In [ ]:
#@title ▶ を押すimport os, globfrom IPython.display import HTML, displayfrom base64 import b64encodeos.chdir('/content/claude-mcp/pipeline')EP    = 7      #@param {type:"integer"}SHORT = False  #@param {type:"boolean"}name = f'ep{EP:02d}{"s" if SHORT else ""}.mp4'if not os.path.exists(name):    print(name, 'がありません。2番のセルを先に回してください')else:    mb = os.path.getsize(name)/1e6    print(name, '%.1f MB' % mb)    if mb < 40:        b = b64encode(open(name,'rb').read()).decode()        display(HTML(f'<video width=320 controls src="data:video/mp4;base64,{b}">'))    else:        print('大きいので再生は省略します')    try:        from google.colab import files        files.download(name)    except Exception as e:        print('自動ダウンロードは失敗。左のフォルダから落としてください:', e)

---### 覚えておくと楽なこと- **声はキャッシュから引いています**（`pipeline/.voicelines`）。  台本の `say=` を書き換えた行だけ、新しく合成されます。  丸ごと作り直しにはなりません。- **モデルは台本側で固定してあります。** 環境変数を付け忘れて  違う声で作り直す事故を防ぐためです。- **Colab は放っておくと切れます。** タブは開いたままにしてください。- 素材（`clips/`）はリポジトリに入っているので、動画素材を  もう一度落としてくる必要はありません。